# TRACE: Memory in Motion
## In-Context Learning with Recurrent Memory
**DataForge 2026 × Pathway Track (PS1)**

This notebook verifies the computational core of TRACE directly from `core/linear_memory.py`.
It reproduces the golden fixtures (`experiments/fixtures.json` and `experiments/expected_results.json`).

> **Explicit Non-Claim:** This educational model instantiates only the additive special case named in BDH-CQ §3.2 ($S_t = S_{t-1} + U_\theta(D_t)$ with $U_\theta(D_t) = v_t k_t^T$). It is not an implementation of BDH or BDH-CQ.

In [1]:
import sys
import os
import json
import numpy as np

# Ensure project root is in path
sys.path.insert(0, os.path.abspath('..'))
from core.linear_memory import LinearMemory, build_codebook
print("LinearMemory successfully imported from core/")

### Experiment 1: The Hook Demo (Orthonormal Keys, d=3)
Writing `red -> 1.0`, `blue -> 2.0`, `green -> 3.0` into a 3×3 matrix.
Querying `green` recovers 3.0 with zero cross-key interference.

In [2]:
d = 3
symbols = ["red", "blue", "green"]
values = {"red": 1.0, "blue": 2.0, "green": 3.0}
cb = build_codebook(symbols, d=d, seed=42)

mem = LinearMemory(d=d)
for s in symbols:
    mem.write(np.array(cb[s]), values[s], label=s)

pred_green = mem.read_scalar(np.array(cb["green"]))
print(f"Final State S_3:\n{mem.state}\n")
print(f"Prediction for 'green': {pred_green:.6f} | Ground Truth: {values['green']:.6f}")
assert np.isclose(pred_green, 3.0, atol=1e-9), "Hook prediction mismatch!"

### Experiment 2: Interference from Overlapping Keys (d=2)
Two non-orthogonal keys $k_A = [1, 0]$ and $k_B = [0.6, 0.8]$.
Theoretical leak coefficient: $(k_A \cdot k_B) / (k_A \cdot k_A) = 0.6$.
Writing $k_A \to 1.0$, then $k_B \to 9.0$.
Querying $k_A$ returns $1.0 + 0.6 \times 9.0 = 6.4$.

In [3]:
key_a = np.array([1.0, 0.0])
key_b = np.array([0.6, 0.8])
leak = LinearMemory.interference_score(key_a, key_b)

mem2 = LinearMemory(d=2)
mem2.write(key_a, 1.0, label="key_A")
mem2.write(key_b, 9.0, label="key_B")

pred_a = mem2.read_scalar(key_a)
theoretical = 1.0 + leak * 9.0

print(f"Derived Leak Coefficient: {leak:.4f}")
print(f"Blended Model Prediction: {pred_a:.4f} | Theoretical: {theoretical:.4f}")
assert np.isclose(pred_a, 6.4, atol=1e-9), "Interference prediction mismatch!"

### Experiment 3: Capacity Dimension Sweep
Testing 4 demonstrations across state dimensions $d \in [2, 3, 4, 5, 6]$.
Verifies the claim: For $d \ge 4$, orthonormal key construction eliminates interference.

In [4]:
symbols_sweep = ["red", "blue", "green", "yellow"]
values_sweep = {"red": 1.0, "blue": 2.0, "green": 3.0, "yellow": 4.0}

for d_val in [2, 3, 4, 5, 6]:
    cb_val = build_codebook(symbols_sweep, d=d_val, seed=100)
    m = LinearMemory(d=d_val)
    for s in symbols_sweep:
        m.write(np.array(cb_val[s]), values_sweep[s])
    
    errs = [abs(m.read_scalar(np.array(cb_val[s])) - values_sweep[s]) for s in symbols_sweep]
    mse = np.mean([e**2 for e in errs])
    print(f"Dimension d={d_val} | MSE: {mse:.6e} | Orthonormal possible: {d_val >= 4}")

### Golden Fixtures Verification Contract
Checking parity against `experiments/expected_results.json`.

In [5]:
with open('../experiments/expected_results.json') as f:
    expected = json.load(f)

assert np.isclose(pred_green, expected['hook_green_prediction'], atol=1e-9)
assert np.isclose(leak, expected['interference_leak_coefficient'], atol=1e-9)
assert np.isclose(pred_a, expected['interference_blended_prediction'], atol=1e-9)
print("All expected results strictly verified against expected_results.json!")